# Download CICIoT2023

Primary ATLAS-IDNet training dataset.

Target folder: `/Volumes/LocalDrive1/Dataset/Security/CICIoT2023`.

Official source: https://www.unb.ca/cic/datasets/iotdataset-2023.html

The CIC download route currently uses a registration form. This notebook supports:

- Kaggle API download, if credentials are configured.
- Manual official download: place the downloaded archive or CSV files in the target folder, then rerun the extract/manifest cells.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import zipfile
from datetime import datetime, timezone

def find_download_src() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd / "src",
        cwd.parent / "src",
        cwd / "implementation" / "download" / "src",
        Path("/Volumes/LocalDrive1/onlyscholar-projects/ATLAS-IDNet/implementation/download/src"),
        Path("/content/ATLAS-IDNet/implementation/download/src"),
    ])
    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / "implementation" / "download" / "src")
        candidates.append(parent / "download" / "src")
    for candidate in candidates:
        if (candidate / "atlas_idnet_download" / "download_config.py").exists():
            return candidate
    raise ModuleNotFoundError(
        "Could not find implementation/download/src. Run this notebook from the repo, "
        "or keep the notebooks and src folders together under implementation/download."
    )


def fallback_download_config():
    dataset_root = Path(os.environ.get("ATLAS_DATASET_ROOT", "/Volumes/LocalDrive1/Dataset/Security"))
    datasets = {
        "CICIoT2023": {
            "target_dir": dataset_root / "CICIoT2023",
            "official_page": "https://www.unb.ca/cic/datasets/iotdataset-2023.html",
            "official_download_form": "https://cicresearch.ca/IOTDataset/CIC_IOT_Dataset2023/",
            "kaggle_slug": "madhavmalhotra/unb-cic-iot-dataset",
            "kaggle_slugs": ["madhavmalhotra/unb-cic-iot-dataset", "himadri07/ciciot2023", "akashdogra/cic-iot-2023"],
        },
        "NF-UQ-NIDS-v2": {
            "target_dir": dataset_root / "NF-UQ-NIDS-v2",
            "official_page": "https://www.cyber.uq.edu.au/project/machine-learning-based-nids-datasets",
            "official_rdm_file": "https://rdm.uq.edu.au/files/e2412450-ef9c-11ed-827d-e762de186848",
            "kaggle_slug": "aryashah2k/nfuqnidsv2-network-intrusion-detection-dataset",
            "kaggle_slugs": ["aryashah2k/nfuqnidsv2-network-intrusion-detection-dataset", "dhoogla/nfuqnidsv2"],
        },
    }
    return dataset_root, datasets


try:
    SRC_DIR = find_download_src()
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))
    from atlas_idnet_download.download_config import DATASET_ROOT, DATASETS
    print("Using shared config:", SRC_DIR)
except ModuleNotFoundError:
    DATASET_ROOT, DATASETS = fallback_download_config()
    print("Shared src package not found; using notebook fallback config.")

DATASET_NAME = "CICIoT2023"
CFG = DATASETS[DATASET_NAME]
TARGET_DIR = CFG["target_dir"]
RAW_DIR = TARGET_DIR / "raw"
EXTRACTED_DIR = TARGET_DIR / "extracted"
MANIFEST_PATH = TARGET_DIR / "download_manifest.json"

for path in [DATASET_ROOT, TARGET_DIR, RAW_DIR, EXTRACTED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Dataset root:", DATASET_ROOT)
print("Target folder:", TARGET_DIR)
print("Official page:", CFG["official_page"])
print("Official download form:", CFG["official_download_form"])

Using shared config: /Volumes/LocalDrive1/onlyscholar-projects/ATLAS-IDNet/implementation/download/src
Dataset root: /Volumes/LocalDrive1/Dataset/Security
Target folder: /Volumes/LocalDrive1/Dataset/Security/CICIoT2023
Official page: https://www.unb.ca/cic/datasets/iotdataset-2023.html
Official download form: https://cicresearch.ca/IOTDataset/CIC_IOT_Dataset2023/


In [2]:
def has_kaggle_credentials() -> bool:
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    return kaggle_json.exists() or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))


def ensure_kaggle_cli() -> None:
    if shutil.which("kaggle"):
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])


def run(cmd):
    print("Running:", " ".join(map(str, cmd)))
    subprocess.check_call(list(map(str, cmd)))


def candidate_kaggle_slugs() -> list[str]:
    slugs = []
    if CFG.get("kaggle_slug"):
        slugs.append(CFG["kaggle_slug"])
    slugs.extend(CFG.get("kaggle_slugs", []))
    return list(dict.fromkeys(slugs))


def ensure_kagglehub() -> None:
    try:
        import kagglehub  # noqa: F401
        return
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])


def materialize_kagglehub_cache(cache_path: Path, raw_dir: Path) -> None:
    raw_dir.mkdir(parents=True, exist_ok=True)
    if cache_path.is_file():
        sources = [cache_path]
    else:
        sources = sorted(cache_path.iterdir())
    for src in sources:
        dst = raw_dir / src.name
        if dst.exists():
            continue
        try:
            dst.symlink_to(src, target_is_directory=src.is_dir())
        except OSError:
            if src.is_dir():
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src, dst)


def try_kagglehub_download() -> bool:
    ensure_kagglehub()
    import kagglehub

    errors = []
    for slug in candidate_kaggle_slugs():
        try:
            print("Trying KaggleHub dataset:", slug)
            cache_path = Path(kagglehub.dataset_download(slug))
            print("KaggleHub cache:", cache_path)
            materialize_kagglehub_cache(cache_path, RAW_DIR)
            return True
        except Exception as exc:
            errors.append(f"{slug}: {exc}")
            print("KaggleHub failed for", slug, "->", exc)
    print("All KaggleHub attempts failed:")
    for error in errors:
        print(" -", error)
    return False


def extract_archives(raw_dir: Path, extracted_dir: Path) -> list[str]:
    extracted = []
    for archive in sorted(raw_dir.iterdir()):
        if archive.suffix.lower() == ".zip":
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(extracted_dir)
            extracted.append(archive.name)
        elif archive.name.endswith((".tar.gz", ".tgz", ".tar")):
            with tarfile.open(archive) as tf:
                tf.extractall(extracted_dir)
            extracted.append(archive.name)
    return extracted


def file_sha256(path: Path, block_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()


def write_manifest(download_method: str, extracted_archives: list[str]) -> None:
    csv_files = sorted(TARGET_DIR.rglob("*.csv"))
    archives = sorted([p for p in RAW_DIR.iterdir() if p.is_file()])
    manifest = {
        "dataset": DATASET_NAME,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_dir": str(TARGET_DIR),
        "download_method": download_method,
        "official_page": CFG["official_page"],
        "official_download_form": CFG["official_download_form"],
        "kaggle_slug": CFG["kaggle_slug"],
        "archive_count": len(archives),
        "csv_count": len(csv_files),
        "extracted_archives": extracted_archives,
        "archives": [
            {"name": p.name, "bytes": p.stat().st_size, "sha256": file_sha256(p)}
            for p in archives
        ],
        "csv_preview": [str(p.relative_to(TARGET_DIR)) for p in csv_files[:20]],
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("Manifest written:", MANIFEST_PATH)
    print("CSV files found:", len(csv_files))

In [3]:
download_method = "existing_files"

existing_csv = list(TARGET_DIR.rglob("*.csv"))
existing_archives = [p for p in RAW_DIR.iterdir() if p.is_file() and p.suffix.lower() in {".zip", ".gz", ".tar"}]

if existing_csv:
    print(f"Found {len(existing_csv)} existing CSV files. Skipping download.")
elif has_kaggle_credentials():
    ensure_kaggle_cli()
    run(["kaggle", "datasets", "download", "-d", CFG["kaggle_slug"], "-p", RAW_DIR, "--force"])
    download_method = "kaggle_api"
elif try_kagglehub_download():
    download_method = "kagglehub"
else:
    raise RuntimeError(
        "No CSV files found, Kaggle CLI credentials are not configured, and KaggleHub failed.\n"
        "Fast fix in a new notebook cell:\n"
        "  import os\n"
        "  os.environ['KAGGLE_USERNAME'] = 'your_username'\n"
        "  os.environ['KAGGLE_KEY'] = 'your_api_key'\n"
        "Then rerun this cell.\n"
        "Manual route: download CICIoT2023 from the official CIC form and place archives/CSV files under "
        f"{TARGET_DIR}. Official form: {CFG['official_download_form']}"
    )

/Users/darsh/miniconda3/envs/research/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Trying KaggleHub dataset: madhavmalhotra/unb-cic-iot-dataset


100%|██████████| 2.77G/2.77G [26:56<00:00, 1.84MB/s] 

Extracting files...


KaggleHub cache: /Users/darsh/.cache/kagglehub/datasets/madhavmalhotra/unb-cic-iot-dataset/versions/1


In [4]:
extracted = extract_archives(RAW_DIR, EXTRACTED_DIR)
if extracted:
    print("Extracted:", extracted)
else:
    print("No archives needed extraction or archives were already extracted.")

write_manifest(download_method, extracted)

No archives needed extraction or archives were already extracted.
Manifest written: /Volumes/LocalDrive1/Dataset/Security/CICIoT2023/download_manifest.json
CSV files found: 0
